## CardioVascular Disease Predication

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from kmodes.kmodes import KModes

In [ ]:
df=pd.read_csv("cardio_train.csv",delimiter=';')
df.drop(columns='id', inplace=True)

In [ ]:
df.head()

In [ ]:
df["age"] = df["age"]/365
df.shape

In [ ]:
numeric_cols = ['height', 'weight', 'ap_hi', 'ap_lo']

plt.figure(figsize=(14, 8))

# Create a boxplot for each numeric column
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')

plt.tight_layout()
plt.show()

In [ ]:
df.isna().sum()

In [ ]:
numeric_cols = ['height', 'weight', 'ap_hi', 'ap_lo']

# Remove outliers outside the 2.5% to 97.5% quantiles
for col in numeric_cols:
    lower = df[col].quantile(0.025)
    upper = df[col].quantile(0.975)
    df[col] = df[col].clip(lower=lower, upper=upper)
    
df.shape

### Feature Reducion

##### Convering Age into bins

In [ ]:

bins = [29, 35, 40, 45, 50, 55, 60, 65]  # Upper bound is exclusive
# labels = ["30-35", "35-40", "40-45", "45-50", "50-55", "55-60", "60-65"]
labels= [0, 1, 2, 3, 4, 5, 6]
df["Age_class"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)


##### Calculating BMI and creating bins of BMI

In [ ]:
df["height"] = df["height"] / 100  # Convert height from cm to meters
df["BMI"] = df["weight"] / (df["height"] ** 2) 

In [ ]:
df.describe()

In [ ]:
bins = [15, 20, 25, 30, 35, 40, 45, 50]  # Upper bound is exclusive
# # labels = ["15-20", "20-25", "25-30", "30-35", "35-40", "40-45", "45-50","50-55"]
labels=[0, 1, 2, 3, 4, 5, 6]

df["BMI_class"] = pd.cut(df["BMI"], bins=bins, labels=labels, right=False)
df = df.drop(columns=["BMI"])


In [ ]:
df.isna().sum()

In [ ]:
df["map"] = (2*df["ap_lo"] + df["ap_hi"])/3

In [ ]:
df["map"].describe()

In [ ]:
bins = [70, 80, 90, 100,110,124]  # Upper bound is exclusive
# labels = ["70-80", "80-90", "90-100", "100-110","110-120"]
labels=[0, 1, 2, 3, 4]

df["MAP_class"] = pd.cut(df["map"], bins=bins, labels=labels, right=False)
df = df.drop(columns=["map"])

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df = df.drop(columns=["age"])
df = df.drop(columns=["ap_lo"])
df = df.drop(columns=["ap_hi"])
df = df.drop(columns=["height"])
df = df.drop(columns=["weight"])

In [ ]:
df['Age_class'] = df['Age_class'].astype(int)
df['BMI_class'] = df['BMI_class'].astype(int)
df['MAP_class'] = df['MAP_class'].astype(int)

In [ ]:
df.head()

### Clustering

In [ ]:

categorical_columns = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 
                       'cardio', 'Age_class', 'BMI_class', 'MAP_class']

# Extract categorical data
cat_data = df[categorical_columns]

# Initialize K-Modes (choose number of clusters, e.g., 3)
km = KModes(n_clusters=2, init='Huang', n_init=5, verbose=1)

# Fit the model and get cluster labels
clusters = km.fit_predict(cat_data)

# Add cluster labels to your original DataFrame
df['cluster'] = clusters


In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(14, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
df_male = df[df["gender"]==1]
df_female = df[df["gender"]==2]

In [ ]:
df_male.shape[0] + df_female.shape[0]

In [ ]:
from sklearn.model_selection import train_test_split

#### Descision Tree

In [ ]:
inputs = df_male.drop('cardio',axis='columns')
inputs.drop('gender', axis='columns', inplace=True)

target=df_male["cardio"]
X_train,X_test,Y_train,Y_test=train_test_split(inputs,target,test_size=0.2)

In [ ]:
X_train.head()

In [ ]:
from sklearn import tree
model=tree.DecisionTreeClassifier()

In [ ]:
model.fit(X_train,Y_train)

In [ ]:
model.score(X_test,Y_test)

### Score from cross validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier


model = DecisionTreeClassifier(random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, inputs,target , cv=kf, scoring='accuracy')


print("Cross-validation scores:", scores)
print("Mean accuracy:", scores.mean())



### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, Y_train)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
Y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(Y_test, Y_pred))

In [ ]:
model = RandomForestClassifier(random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, inputs,target , cv=kf, scoring='accuracy')


print("Cross-validation scores:", scores)
print("Mean accuracy:", scores.mean())

### XG Boost

In [ ]:
from xgboost import XGBClassifier
xg = XGBClassifier(use_label_encoder=False, eval_metric='logloss',enable_categorical=True, eta=0.01)
xg.fit(X_train, Y_train)

In [ ]:
Y_pred = xg.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(Y_test, Y_pred))
# print("\nClassification Report:\n", classification_report(Y_test, Y_pred))

In [ ]:

model = XGBClassifier(random_state=42)

# Step 2: Define K-Fold (e.g., 5 splits)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Step 3: Cross-validation scoring
scores = cross_val_score(model, inputs, target, cv=kf, scoring='accuracy')

# Step 4: Print results
print("Cross-validation scores:", scores)
print("Mean accuracy:", scores.mean())


### MLP

In [ ]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.01,
    alpha=0.0001,
    early_stopping=True,
    validation_fraction=0.1,
    max_iter=500,
    random_state=42
)
mlp.fit(X_train, Y_train)

In [ ]:
Y_pred = mlp.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(Y_test, Y_pred))

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(mlp, inputs, target, cv=kf, scoring='accuracy')

print("Cross-validation scores:", scores)
print("Mean accuracy:", scores.mean())

#### MLP with GridSearchCV

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

# Define the base model
mlp = MLPClassifier(
    early_stopping=True,
    validation_fraction=0.1,
    max_iter=500,
    random_state=42
)

# Define parameter grid
param_grid = {
    'hidden_layer_sizes': [(64,), (128,), (128, 64), (128, 64, 32)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'learning_rate_init': [0.001, 0.01],
    'alpha': [0.0001, 0.001],  # L2 regularization
}

# Setup GridSearchCV
grid = GridSearchCV(
    mlp,
    param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Fit the model
grid.fit(X_train, Y_train)

# Print results
print("? Best Accuracy:", grid.best_score_)
print("?? Best Params:", grid.best_params_)


### LGBM

In [ ]:
import lightgbm as lgb

In [ ]:
train_data = lgb.Dataset(X_train, label=Y_train, categorical_feature='auto')
test_data = lgb.Dataset(X_test, label=Y_test, categorical_feature='auto')


params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbose': -1
}
num_round = 1000
bst = lgb.train(params, train_data, num_round, valid_sets=[
                test_data])

In [ ]:
y_pred_probs = bst.predict(X_test)         # Probabilities
y_pred = (y_pred_probs > 0.5).astype(int)     # Convert to class labels

from sklearn.metrics import accuracy_score
acc = accuracy_score(Y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import KFold, cross_val_score

# Use the sklearn wrapper
model = LGBMClassifier(random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, inputs, target, cv=kf, scoring='accuracy')

print("Cross-validation scores:", scores)
print("Mean accuracy:", scores.mean())


### After Hyper Parameter tunning

### CatBoost

In [ ]:
from catboost import CatBoostClassifier

# Define categorical features (as indices)
cat_features = [X_train.columns.get_loc(col) for col in [
 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'Age_class', 'BMI_class', 'MAP_class'
]]

# Initialize CatBoostClassifier with default parameters
model = CatBoostClassifier(verbose=0, random_state=42)

# Train the model
model.fit(X_train, Y_train, cat_features=cat_features)


In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(Y_test, y_pred)
print(f"CatBoost Accuracy: {acc:.4f}")


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, inputs, target, cv=kf, scoring='accuracy')

print("Cross-validation scores:", scores)
print("Mean accuracy:", scores.mean())

#### IENN